# Phase 3: Piper TTS (VITS) Cloud Training & ONNX Export for Santhali
### Fast, Lightweight Voice Synthesis for Offline Android (`sat_Olck`)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AshrafGalaxy/Vernacular_Pedagogy/blob/main/notebooks/colab_phase3_piper_tts.ipynb)

**Objective:**
1. Train or fine-tune **Piper TTS** (VITS-based acoustic + vocoder neural architecture) on standardized Santhali Common Voice audio (`metadata.csv` + 16kHz Mono WAVs).
2. Export the trained model to **ONNX INT8 / FP16 format** (~30 MB) with accompanying configuration JSON (`model.onnx.json`) for zero-latency classroom playback on low-end Android devices.

## 1. Environment Setup & Piper Training Dependencies

In [ ]:
# Check GPU availability
!nvidia-smi

# Install PyTorch Lightning, ONNX, and Piper training suite
!pip install -q torch torchvision torchaudio pytorch-lightning==1.9.5 torchmetrics==0.11.4 onnx onnxruntime
!git clone https://github.com/rhasspy/piper.git /content/piper
!cd /content/piper/src/python && pip install -q -e .

## 2. Ingest Standardized Voice Bank Artifact
Load the 16kHz Mono WAVs and `metadata.csv` generated from Phase 1.

In [ ]:
import os

# Mount Google Drive or unpack downloaded voice bank archive
VOICEBANK_ARCHIVE = "/content/santali_piper_voicebank_16k.tar.gz"
TARGET_DATA_DIR = "/content/dataset"

os.makedirs(TARGET_DATA_DIR, exist_ok=True)

if os.path.exists(VOICEBANK_ARCHIVE):
    !tar -xzf {VOICEBANK_ARCHIVE} -C {TARGET_DATA_DIR}
    print("Unpacked voice bank dataset successfully!")
else:
    print("Please provide /content/santali_piper_voicebank_16k.tar.gz from Phase 1 audio prep.")

!ls -lh {TARGET_DATA_DIR}

## 3. Preprocess Dataset & Character Phoneme Alignment
Piper converts text into phoneme IDs. For Santhali Ol Chiki, each Unicode glyph (`U+1C50 - U+1C7F`) maps deterministically to its canonical acoustic token.

In [ ]:
%%bash
# Format dataset into Piper training format
cd /content/piper/src/python
python3 -m piper_train.preprocess \
  --language sat \
  --input-dir /content/dataset \
  --output-dir /content/piper_training_dir \
  --dataset-format ljspeech \
  --single-speaker \
  --sample-rate 16000

## 4. Download Pre-trained Piper Base VITS Checkpoint
Warm-starting from a pre-trained Indic / multilingual checkpoint reduces training time from days to 1–2 hours.

In [ ]:
# Fetch official Piper VITS base checkpoint
!wget -q -O /content/piper_base.ckpt https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/en/en_US/lessac/medium/epoch%3D2164-step%3D1355540.ckpt
print("Base checkpoint downloaded ready for fine-tuning.")

## 5. Fine-Tune Piper TTS Model

In [ ]:
%%bash
cd /content/piper/src/python
python3 -m piper_train \
  --dataset-dir /content/piper_training_dir \
  --accelerator gpu \
  --devices 1 \
  --batch-size 16 \
  --validation-split 0.05 \
  --checkpoint-epochs 100 \
  --max-epochs 3000 \
  --resume_from_checkpoint /content/piper_base.ckpt

## 6. Export Trained Model to ONNX for Android Edge Deployment

In [ ]:
%%bash
cd /content/piper/src/python
# Find latest checkpoint
LATEST_CKPT=$(ls -t /content/piper_training_dir/lightning_logs/version_*/checkpoints/*.ckpt | head -n 1)
echo "Exporting checkpoint: $LATEST_CKPT to ONNX..."

python3 -m piper_train.export_onnx \
  "$LATEST_CKPT" \
  /content/sat_piper_model.onnx

# Copy model configuration json
cp /content/piper_training_dir/config.json /content/sat_piper_model.onnx.json
ls -lh /content/sat_piper_model.onnx*

## 7. Test Inference & Audio Verification

In [ ]:
import onnxruntime as ort
import numpy as np
import json
from IPython.display import Audio

# Load ONNX session
onnx_path = "/content/sat_piper_model.onnx"
if os.path.exists(onnx_path):
    session = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
    print("ONNX TTS Model loaded successfully on CPU!")
    print("Inputs:", [i.name for i in session.get_inputs()])
    print("Outputs:", [o.name for o in session.get_outputs()])
else:
    print("Run training and export to generate the ONNX artifact.")

## 8. Package & Export for Android Assets

In [ ]:
%%bash
cd /content
if [ -f "sat_piper_model.onnx" ]; then
    tar -czf sat_piper_onnx_android.tar.gz sat_piper_model.onnx sat_piper_model.onnx.json
    ls -lh sat_piper_onnx_android.tar.gz
    echo "Ready to copy into Android app assets/models/tts/ !"
fi